In [1]:
import sys
from collections import defaultdict
from itertools import combinations
from pathlib import Path
from statistics import correlation, mean

import litellm
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import kendalltau, rankdata
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import LassoCV, LinearRegression, RidgeCV
from sklearn.model_selection import LeaveOneOut
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from tabulate import tabulate
from wordfreq import zipf_frequency
from words import get_words

# Add the repo root to sys.path so we can import our modules
repo_root = Path.cwd().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from GenerateKeywordCards2.async_rng import AsyncRng  # noqa: E402
from GenerateKeywordCards2.check_familiarity import (  # noqa: E402
    check_familiarity_async,  # noqa: E402
)

# from asyncio import TaskGroup  <-- Doesn't work in Jupyter, use CompatTaskGroup instead.
from GenerateKeywordCards2.compat_task_group import (  # noqa: E402
    CompatTaskGroup as TaskGroup,  # noqa: E402
)
from GenerateKeywordCards2.manual_ratings import (  # noqa: E402
    add_manual_ratings_words,
    get_manual_ratings,
)
from GenerateKeywordCards2.pip_audit_with_urls import pip_audit_with_urls  # noqa: E402
from GenerateKeywordCards2.rate_words import RatingsByWord  # noqa: E402
from GenerateKeywordCards2.rate_words_with_variations import (  # noqa: E402
    ModelParams,
    VariationParams,
    VariationView,
    iter_variation_views,
    rate_words_with_variations,
)

In [2]:
# For some reason, the Ruff: Format Imports command fails with "unspecified reason" if these imports are in the same cell as the other imports.
from GenerateKeywordCards2.rate_words_with_variations import (  # noqa: E402
    combine_results,
    filter_results,
)

In [3]:
litellm.cache = litellm.Cache(type="disk")


def get_cache_count() -> int:
    disk_cache_object = litellm.cache.cache.disk_cache
    return len(disk_cache_object)


initial_cache_count = get_cache_count()

# Candidate Words

In [8]:
all_words = get_words()
print(f"{len(all_words):,} total words: ... {all_words[-25:]}")

existing_keywords_file = (
    repo_root / "GenerateKeywordCards" / "CloverExistingKeywords.csv"
)
existing_keywords = existing_keywords_file.read_text(encoding="utf-8-sig").splitlines()
existing_keywords = [w.lower() for w in existing_keywords]
existing_keywords_set = set(existing_keywords)
candidate_words = [w for w in all_words if w not in existing_keywords_set]
print(f"{len(candidate_words):,} candidate words with existing keywords removed")

50,000 total words: ... ['rerouted', 'resourcefulness', 'resto', "rice's", 'richman', 'rie', 'rigors', 'riptide', 'rivas', 'rocca', 'rolando', 'romancing', 'roti', 'rungs', 'rushdie', 'sah', 'salafi', 'sani', 'satanism', 'schell', 'screensaver', 'sdgs', 'seder', 'sek', 'selectable']
49,124 candidate words with existing keywords removed
